# Student 2 Individual Notebook

Kaitlin Luu

Student 2: TWO ensemble methods: one Bagging-type model (e.g., Random Forest) AND one Boosting-type model (e.g., Gradient Boosting).

## Data Preparation

In [4]:
# read in data

import pandas as pd
df = pd.read_csv("../../Data/training.csv")
df.head()

,rev_util,age,late_30_59,debt_ratio,monthly_inc,open_credit,late_90,real_estate,late_60_89,dependents,dlq_2yrs
0,0.793106,35.0,1.0,0.098301,3000.0,3.0,0.0,0.0,1.0,4.0,1
1,1.000000,43.0,4.0,0.857017,2272.0,5.0,0.0,2.0,0.0,2.0,1
2,0.772902,78.0,0.0,2.414651,2811.0,21.0,0.0,1.0,0.0,0.0,0
3,2.099010,43.0,0.0,0.033319,2400.0,1.0,0.0,0.0,0.0,0.0,0
4,0.266185,49.0,0.0,0.179646,40000.0,16.0,0.0,3.0,0.0,3.0,0


In [5]:
# prepare data

X = df.drop(columns=['dlq_2yrs'])
y = df['dlq_2yrs']
print(X.head())
print(y)

   rev_util   age  late_30_59  debt_ratio  monthly_inc  open_credit  late_90  \
0  0.793106  35.0         1.0    0.098301       3000.0          3.0      0.0   
1  1.000000  43.0         4.0    0.857017       2272.0          5.0      0.0   
2  0.772902  78.0         0.0    2.414651       2811.0         21.0      0.0   
3  2.099010  43.0         0.0    0.033319       2400.0          1.0      0.0   
4  0.266185  49.0         0.0    0.179646      40000.0         16.0      0.0   

   real_estate  late_60_89  dependents  
0          0.0         1.0         4.0  
1          2.0         0.0         2.0  
2          1.0         0.0         0.0  
3          0.0         0.0         0.0  
4          3.0         0.0         3.0  
0        1
1        1
2        0
3        0
4        0
        ..
13366    0
13367    1
13368    0
13369    1
13370    0
Name: dlq_2yrs, Length: 13371, dtype: int64


In [6]:
# 80/20 split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1234
)

print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (10696, 10)  Test: (2675, 10)


## Model 1: Random Forest (Bagging)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(random_state=1234, n_jobs=-1))
])

rf_grid_params = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [5, 10, None],
    'model__min_samples_leaf': [1, 5],
    'model__max_features': ['sqrt', 0.5]
}

rf_grid = GridSearchCV(rf_pipe, rf_grid_params, cv=cv, scoring='roc_auc',
                       n_jobs=1, refit=True, verbose=1)

# fit on training data
rf_grid.fit(X_train, y_train)

print("Best Random Forest parameters:", rf_grid.best_params_)
print("Best mean CV ROC-AUC:", round(rf_grid.best_score_, 4))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Random Forest parameters: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 100}
Best mean CV ROC-AUC: 0.8513


## Model 2: Gradient Boosting (Boosting)

In [11]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', GradientBoostingClassifier(random_state=1234))
])

gb_grid_params = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3, 4],
    'model__subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(gb_pipe, gb_grid_params, cv=cv, scoring='roc_auc',
                       n_jobs=-1, refit=True, verbose=1)
gb_grid.fit(X_train, y_train)

print("Best Gradient Boosting parameters:", gb_grid.best_params_)
print("Best mean CV ROC-AUC:", round(gb_grid.best_score_, 4))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Gradient Boosting parameters: {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best mean CV ROC-AUC: 0.8535
